In [3]:
# load_data.py  (second file)
from pathlib import Path
import tensorflow as tf

WORK_DIR = Path("animal")   # must match the first file
IMG_SIZE = (64, 64)
SEED     = 42
BATCH    = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    str(WORK_DIR / "train"), image_size=IMG_SIZE, batch_size=BATCH,
    seed=SEED, shuffle=True)
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(WORK_DIR / "val"), image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
test_ds = tf.keras.utils.image_dataset_from_directory(
    str(WORK_DIR / "test"), image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)

print("Classes:", train_ds.class_names)

# Scale pixels to 0-1 for the custom model A
normalize = tf.keras.layers.Rescaling(1.0 / 255)
train_ds_n = train_ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE)
val_ds_n   = val_ds.map(lambda x, y: (normalize(x), y),   num_parallel_calls=tf.data.AUTOTUNE)
test_ds_n  = test_ds.map(lambda x, y: (normalize(x), y),  num_parallel_calls=tf.data.AUTOTUNE)

Found 2100 files belonging to 3 classes.
Found 450 files belonging to 3 classes.
Found 450 files belonging to 3 classes.
Classes: ['cats', 'dogs', 'snakes']


In [4]:
from tensorflow.keras import layers, models
import time

tf.keras.utils.set_random_seed(SEED)        # reproducible weight initialisation
NUM_CLASSES = len(train_ds.class_names)
AUTOTUNE = tf.data.AUTOTUNE

# Keep decoded images in memory and prepare the next batch while the GPU trains (faster epochs).
# Training data: cache single images, then re-shuffle every epoch and re-batch.
train_ds_n = (train_ds_n.unbatch().cache()
              .shuffle(2000, seed=SEED, reshuffle_each_iteration=True)
              .batch(BATCH).prefetch(AUTOTUNE))
val_ds_n   = val_ds_n.cache().prefetch(AUTOTUNE)
test_ds_n  = test_ds_n.cache().prefetch(AUTOTUNE)

def build_model_a(input_shape=(64, 64, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, kernel_size=3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs, name="Model_A_StandardCNN")

model_a = build_model_a()

In [ ]:
def conv_params(k, c_in, c_out):  return k * k * c_in * c_out + c_out
def dense_params(n_in, n_out):     return n_in * n_out + n_out
def conv_macs(h, w, k, c_in, c_out): return h * w * k * k * c_in * c_out   # 'same' padding, stride 1

rows = []   # (layer, params, MACs)
c_in, size = 3, 64
for f in [32, 64, 128, 256]:
    rows.append((f"Conv2D 3x3, {f} ({size}x{size})", conv_params(3, c_in, f), conv_macs(size, size, 3, c_in, f)))
    c_in, size = f, size // 2
rows.append(("Dense 128", dense_params(256, 128), 256 * 128))
rows.append((f"Dense {NUM_CLASSES}", dense_params(128, NUM_CLASSES), 128 * NUM_CLASSES))

print(f"{'Layer':<30}{'Params':>12}{'MACs':>16}")
for name, p, m in rows:
    print(f"{name:<30}{p:>12,}{m:>16,}")
total_p = sum(r[1] for r in rows); total_m = sum(r[2] for r in rows)
print(f"{'TOTAL':<30}{total_p:>12,}{total_m:>16,}")
print("Keras count:", f"{model_a.count_params():,}", "| match:", total_p == model_a.count_params())

In [ ]:
class EpochTimer(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self._t = time.time()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.time() - self._t)

EPOCHS = 20
model_a.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                loss="sparse_categorical_crossentropy",
                metrics=["accuracy"])

timer_a = EpochTimer()
history_a = model_a.fit(train_ds_n, validation_data=val_ds_n, epochs=EPOCHS, callbacks=[timer_a])

# first epoch includes one-off setup (graph building, caching), so report the mean of the rest
print(f"Average time per epoch (epochs 2-{EPOCHS}): {np.mean(timer_a.times[1:]):.2f} s")

In [ ]:
def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history.history["loss"], label="train");     ax[0].plot(history.history["val_loss"], label="validation")
    ax[0].set_title(f"{title} – loss");     ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(history.history["accuracy"], label="train"); ax[1].plot(history.history["val_accuracy"], label="validation")
    ax[1].set_title(f"{title} – accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout()
    plt.savefig(WORK_DIR / f"{title.replace(' ', '_')}_curves.png", dpi=150)
    plt.show()

plot_history(history_a, "Model A")